# Notebook 04 - Mini Benchmark Demo

Đo speed + accuracy nhanh + biểu đồ so sánh.

In [ ]:
import os, sys, time
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
sys.path.insert(0, '..')
import torch
import numpy as np
import matplotlib.pyplot as plt
from ml2.u2net.model import U2NETp

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = U2NETp().to(device).eval()
x = torch.randn(1, 3, 320, 320, device=device)

# Warmup
with torch.no_grad():
    for _ in range(5):
        _ = model(x)
if device.type == 'mps':
    torch.mps.synchronize()

times = []
for _ in range(30):
    t = time.perf_counter()
    with torch.no_grad():
        _ = model(x)
    if device.type == 'mps':
        torch.mps.synchronize()
    times.append((time.perf_counter() - t) * 1000)

print(f'Median: {np.median(times):.1f}ms | p95: {np.percentile(times, 95):.1f}ms | FPS: {1000 / np.median(times):.1f}')

In [ ]:
# Plot histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(times, bins=15, color='steelblue', edgecolor='black')
ax.axvline(np.median(times), color='red', linestyle='--', label=f'Median {np.median(times):.1f}ms')
ax.set_xlabel('Latency (ms)'); ax.set_ylabel('Count')
ax.set_title('U²-Netp inference latency'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Mock comparison (replace với KPI thật sau khi train)
import matplotlib.pyplot as plt
models = ['rembg', 'U2-Netp', 'YOLO11n-seg']
iou_mock = [0.78, 0.85, 0.82]
fps_mock = [8, 25, 40]
size_mock = [176, 4.7, 6]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(models, iou_mock, color=['gray', 'steelblue', 'seagreen'])
axes[0].set_title('mIoU'); axes[0].set_ylim(0, 1)
axes[1].bar(models, fps_mock, color=['gray', 'steelblue', 'seagreen'])
axes[1].set_title('FPS (MPS)')
axes[2].bar(models, size_mock, color=['gray', 'steelblue', 'seagreen'])
axes[2].set_title('Model size (MB)'); axes[2].set_yscale('log')
plt.tight_layout(); plt.show()